## Training dataset, independent dataset split

In [ ]:
# read the features df
features_df = pd.read_csv("features_df.csv")
features_df

In [1]:
# Training dataset, independent dataset split

In [ ]:
# check the logic in notebook 1 analysis and figure
# if the uniprot ID have been used in previous dataset, it should be in the training dataset
# if the protein is augmentated from a uniprot ID which are in the previous dataset, it should be in the training dataset

# balance training dataset and independent dataset (80% training and 20% independent dataset)

In [2]:
import pandas as pd
pred_dataset_name = 'data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx'
finder_dataset_name = "data/NIHMS1881042-supplement-Supplementary_table_dataset_2.xlsx"
curated_dataset_name = "data/20240518final_merge_data_with_high_1433pre_mapped_to_human.csv"

In [3]:
# the data in data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx have 4 tab, combine them into one
file_path = pred_dataset_name
excel_file = pd.ExcelFile(file_path)

# Let's get the sheet names
sheet_names = excel_file.sheet_names

# Create an empty list to hold dataframes
dfs = []

# Loop through the sheet names and read each sheet into a dataframe
for sheet in sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    # Optionally, you can add a column indicating the sheet name, useful for tracking the data source
    df['Source_Sheet'] = sheet
    dfs.append(df)

# Concatenate all the dataframes
pred_dataset_df = pd.concat(dfs, ignore_index=True)
pred_dataset_df = pred_dataset_df[pred_dataset_df['PMID'] != '*Likely NEG sites ']
pred_dataset_df = pred_dataset_df[["Uniprot ID","Site","Residue"]]
pred_dataset_df["Site"] = pred_dataset_df["Site"].astype(int)
pred_dataset_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx'

In [4]:
# 14-3-3 finder dataset preprocess
finder_dataset_df = pd.read_excel(finder_dataset_name)
finder_dataset_df = finder_dataset_df[["Uniprot ID", 'Site',"Residue"]]
finder_dataset_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/NIHMS1881042-supplement-Supplementary_table_dataset_2.xlsx'

In [ ]:
import re
previous_datset_df = pd.concat([pred_dataset_df, finder_dataset_df], ignore_index=True)
previous_datset_df

In [ ]:
# put the uniprot ID into a set
uniprotid_dict = set(previous_datset_df["Uniprot ID"])
uniprotid_dict

In [ ]:
training_dataset_df =  features_df[features_df['uniqueID'].isin(uniprotid_dict)]
training_dataset_df["label"].value_counts()

In [ ]:
remaining_dataset_df =  features_df[~features_df['uniqueID'].isin(uniprotid_dict)]
remaining_dataset_df_positive = remaining_dataset_df[remaining_dataset_df['label']==1]
remaining_dataset_df_negative = remaining_dataset_df[remaining_dataset_df['label']==0]

In [ ]:
# add more data into training_dataset_df
# for positive data, add uniprot and augmentated data to about 1200 positive sample

def sample_data(df_source, column_name, target_length):
    # Initialize an empty DataFrame B
    df_target = pd.DataFrame(columns=df_source.columns)
    
    # Keep track of the unique values already processed to avoid duplication
    processed_values = set()

    while len(df_target) < target_length and not df_source.empty:
        # Randomly sample one row from the source DataFrame
        sampled_row = df_source.sample(1)
        k_value = sampled_row.iloc[0][column_name]

        # Check if this value has already been processed
        if k_value not in processed_values:
            # Get all rows with the same value in the specified column
            rows_with_k = df_source[df_source[column_name] == k_value]

            # Add these rows to the target DataFrame
            df_target = pd.concat([df_target, rows_with_k], ignore_index=True)

            # Update the set of processed values
            processed_values.add(k_value)

            # Drop these rows from the source DataFrame to avoid resampling
            df_source = df_source[df_source[column_name] != k_value]

    # Ensure the target DataFrame doesn't exceed the required row count
#     if len(df_target) > target_length:
#         df_target = df_target.iloc[:target_length]

    return df_target


column_name = 'augmentated from uniprot ID'
target_length = 1200 - len(training_dataset_df[training_dataset_df['label']==1])
df_sampling_training_positive = sample_data(remaining_dataset_df_positive, column_name, target_length)
df_sampling_training_positive

In [ ]:
# for the independent negative data, add to 1200, also make sure the sample following whole distribution
sample_size = 1200
df_labeled_and_augmented_negative_data = pd.read_csv("data/labeled_and_augmented_negative_data.csv")
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
df_sampling_training_negative = pd.DataFrame() 
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = df_labeled_and_augmented_negative_data[df_labeled_and_augmented_negative_data["stratification_label"] == index]
        sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_training_negative) == 0:
            df_sampling_training_negative = sampled_df
        else:
            df_sampling_training_negative = pd.concat([df_sampling_training_negative, sampled_df], axis=0)

In [ ]:
training_dataset_df = pd.concat([training_dataset_df, df_sampling_training_positive, df_sampling_training_negative], axis=0)

In [ ]:
training_dataset_df.shape()

In [ ]:
training_dataset_df["label"].value_counts

In [ ]:
# all the remaining data are below to indepnedent dataset, about 300 positive and 300 negative sample
# sampling from the remaining negative sample to get 300 negative sample
sample_size = 300
df_labeled_and_augmented_negative_data = pd.read_csv("data/labeled_and_augmented_negative_data.csv")
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
df_sampling_indepnedent_negative = pd.DataFrame()
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = df_labeled_and_augmented_negative_data[df_labeled_and_augmented_negative_data["stratification_label"] == index]
        df_tmp = df_tmp[(~df_tmp["uniprot ID"].isin(uniprotid_dict))&(~df_tmp["uniprot ID"].isin(set(training_dataset_df["uniprot id"])))]
        sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_indepnedent_negative) == 0:
            df_sampling_indepnedent_negative = sampled_df
        else:
            df_sampling_indepnedent_negative = pd.concat([df_sampling_indepnedent_negative, sampled_df], axis=0)

In [ ]:
# all the remaining positive data as the 300 positive independent dataset
df_sampling_indepnedent_positive

In [ ]:
independent_dataset_df = pd.concat([df_sampling_indepnedent_positive, df_sampling_indepnedent_negative], axis=0)

### 4.3	Feature selection 

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

### 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.

task 47: download and curate prediction data from clinvar

task 48: prediction

task 49: figure

task: model update